# Notebook 01: Preprocesamiento y Limpieza de Datos
**Proyecto:** Hit Predictor & Trendy Dashboard — Grupo 5

## Objetivo
Unificar canciones de alta y baja popularidad para analizar qué características musicales se asocian con una canción popular o un hit. El resultado tendrá **una fila por canción**, identificada por `track_id`, y conservará las variables musicales necesarias para análisis y visualizaciones.

### Reglas de limpieza acordadas
1. `track_id` define una canción única.
2. Si un `track_id` aparece más de una vez, se conserva la primera aparición. Como se carga primero `high_popularity`, esta fuente tiene prioridad. Las diferencias de playlist no forman parte de la unidad de análisis.
3. Una canción es un hit (`is_hit = 1`) si `track_popularity >= 70`; en caso contrario, `is_hit = 0`.
4. Se eliminan filas sin variables críticas o con valores fuera de los rangos válidos definidos en este notebook.
5. Los textos se leen y exportan en UTF-8, conservando caracteres en español, inglés y otros idiomas.

In [1]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Librerias cargadas correctamente.')

Librerias cargadas correctamente.


## 1. Carga y validación de las fuentes
Los archivos se leen en UTF-8. Antes de unirlos, verificamos que tengan exactamente el mismo esquema para no mezclar columnas incompatibles.

In [2]:
DATA_DIR = Path('../data')
PATH_HIGH = DATA_DIR / 'high_popularity_spotify_data.csv'
PATH_LOW = DATA_DIR / 'low_popularity_spotify_data.csv'

df_high = pd.read_csv(PATH_HIGH, sep=',', encoding='utf-8')
df_low = pd.read_csv(PATH_LOW, sep=',', encoding='utf-8')

if set(df_high.columns) != set(df_low.columns):
    raise ValueError('Las fuentes no tienen el mismo conjunto de columnas.')

print(f'Alta popularidad: {df_high.shape[0]:,} filas y {df_high.shape[1]} columnas')
print(f'Baja popularidad: {df_low.shape[0]:,} filas y {df_low.shape[1]} columnas')
print('Esquema validado: ambas fuentes tienen las mismas columnas.')

Alta popularidad: 1,686 filas y 29 columnas
Baja popularidad: 3,145 filas y 29 columnas
Esquema validado: ambas fuentes tienen las mismas columnas.


## 2. Normalización de texto y consolidación
Normalizamos los campos de texto sin recodificarlos: esto preserva acentos, alfabetos no latinos y emojis. Después descartamos IDs vacíos y deduplicamos por `track_id`.

In [3]:
TEXT_COLUMNS = ['track_name', 'track_artist', 'track_album_name', 'playlist_name']

def normalize_text(value):
    if isinstance(value, str):
        return unicodedata.normalize('NFC', value).strip()
    return value

for frame in (df_high, df_low):
    for column in TEXT_COLUMNS:
        if column in frame.columns:
            frame[column] = frame[column].map(normalize_text)

    invalid_text = sum(
        frame[column].dropna().astype(str).str.contains(chr(0xFFFD), regex=False).sum()
        for column in TEXT_COLUMNS if column in frame.columns
    )
    if invalid_text:
        raise ValueError('Se detectaron caracteres de reemplazo; revise la codificacion de origen.')

df_high['_source'] = 'high_popularity'
df_low['_source'] = 'low_popularity'
df_combined = pd.concat([df_high, df_low], ignore_index=True)
registros_iniciales = len(df_combined)

df_combined['track_id'] = (
    df_combined['track_id'].astype('string').str.strip().replace('', pd.NA)
)
ids_invalidos = df_combined['track_id'].isna().sum()
df_valid_id = df_combined.dropna(subset=['track_id']).copy()

duplicados_removidos = df_valid_id.duplicated(subset='track_id', keep='first').sum()
df_clean = df_valid_id.drop_duplicates(subset='track_id', keep='first').copy()

print(f'Registros iniciales: {registros_iniciales:,}')
print(f'IDs vacios o invalidos eliminados: {ids_invalidos:,}')
print(f'Duplicados por track_id eliminados: {duplicados_removidos:,}')
print(f'Canciones unicas tras consolidar: {len(df_clean):,}')

Registros iniciales: 4,831
IDs vacios o invalidos eliminados: 0
Duplicados por track_id eliminados: 336
Canciones unicas tras consolidar: 4,495


## 3. Validación de calidad e ingeniería de características
Convertimos las métricas a valores numéricos, eliminamos filas con datos críticos faltantes o valores imposibles, y creamos `duration_min` e `is_hit`. Estas validaciones aseguran que los gráficos y análisis posteriores no se distorsionen por registros defectuosos.

In [4]:
NUMERIC_COLUMNS = [
    'track_popularity', 'duration_ms', 'danceability', 'energy', 'valence',
    'tempo', 'loudness', 'acousticness', 'liveness', 'speechiness',
    'instrumentalness'
]
CRITICAL_COLUMNS = [
    'track_id', 'track_name', 'track_artist', 'track_popularity', 'duration_ms',
    'danceability', 'energy', 'valence', 'tempo', 'loudness'
]

for column in NUMERIC_COLUMNS:
    df_clean[column] = pd.to_numeric(df_clean[column], errors='coerce')

missing_critical = df_clean[CRITICAL_COLUMNS].isna().any(axis=1)
invalid_ranges = (
    ~df_clean['track_popularity'].between(0, 100)
    | (df_clean['duration_ms'] <= 0)
    | ~df_clean['danceability'].between(0, 1)
    | ~df_clean['energy'].between(0, 1)
    | ~df_clean['valence'].between(0, 1)
    | ~df_clean['acousticness'].between(0, 1)
    | ~df_clean['liveness'].between(0, 1)
    | ~df_clean['speechiness'].between(0, 1)
    | ~df_clean['instrumentalness'].between(0, 1)
    | (df_clean['tempo'] <= 0)
)

rows_to_remove = missing_critical | invalid_ranges
print(f'Filas con valores críticos faltantes: {missing_critical.sum():,}')
print(f'Filas con valores fuera de rango: {(invalid_ranges & ~missing_critical).sum():,}')
print(f'Total de filas eliminadas por calidad: {rows_to_remove.sum():,}')

df_clean = df_clean.loc[~rows_to_remove].copy()
df_clean['duration_min'] = (df_clean['duration_ms'] / 60_000).round(2)
df_clean['is_hit'] = (df_clean['track_popularity'] >= 70).astype('int8')

assert df_clean['track_id'].is_unique, 'Persisten track_id duplicados.'
assert df_clean[CRITICAL_COLUMNS].notna().all().all(), 'Persisten nulos críticos.'

distribution = df_clean['is_hit'].value_counts().reindex([1, 0], fill_value=0)
print('\n--- RESUMEN FINAL ---')
print(f'Canciones listas para analisis: {len(df_clean):,}')
print(f'Hits (is_hit = 1): {distribution[1]:,} ({distribution[1] / len(df_clean):.1%})')
print(f'No hits (is_hit = 0): {distribution[0]:,} ({distribution[0] / len(df_clean):.1%})')

Filas con valores críticos faltantes: 1
Filas con valores fuera de rango: 0
Total de filas eliminadas por calidad: 1

--- RESUMEN FINAL ---
Canciones listas para analisis: 4,494
Hits (is_hit = 1): 1,227 (27.3%)
No hits (is_hit = 0): 3,267 (72.7%)


## 4. Exportación del dataset limpio
El archivo final se guarda en UTF-8. La columna técnica `_source` se elimina porque sólo se utilizó para trazabilidad durante la consolidación; las variables musicales y la etiqueta `is_hit` permanecen disponibles para gráficos y modelamiento.

In [5]:
OUTPUT_PATH = DATA_DIR / 'spotify_clean.csv'
df_clean = df_clean.drop(columns='_source')
df_clean.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print(f'Dataset limpio exportado en: {OUTPUT_PATH.resolve()}')
print(f'Columnas disponibles para analisis: {df_clean.shape[1]}')

Dataset limpio exportado en: C:\Users\JUANP\OneDrive\Escritorio\Grupo5_-main\data\spotify_clean.csv
Columnas disponibles para analisis: 31
